In [1]:
import ast
import os
import re
import json
import subprocess
import shutil
from dataclasses import dataclass, asdict
from pathlib import Path
from urllib.parse import urlparse

In [2]:
def clone_repo(url: str, dest_root: str='./temp/repos/')->Path:
    """"Clone a github repo and return the local path.
    re-Clones the cleany if the destination already exist."""

    parsed=urlparse(url)
    repo_name=Path(parsed.path).stem #owner/name.git -> name
    dest=Path(dest_root)/repo_name

    if dest.exists():
        shutil.rmtree(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)

    result=subprocess.run(
        ['git', 'clone', '--depth', '1', url, str(dest)],
        capture_output=True, text=True,
    )

    if result.returncode!=0:
        raise RuntimeError(f"git clone failed: {result.stderr.strip()}")
    
    return dest, repo_name

In [3]:
repo_url="https://github.com/shreeragkh/Hybrid-Search-RAG"

repo_path, repo_name=clone_repo(repo_url)
print(f"cloned {repo_name} -> {repo_path}")
print(f"Files on disk: {sum(1 for _ in repo_path.rglob('*') if _.is_file())}")

cloned Hybrid-Search-RAG -> temp/repos/Hybrid-Search-RAG
Files on disk: 71


In [4]:
CODE_ONLY_MAP = {
    ".py": "python", ".js": "javascript", ".jsx": "javascript",
    ".ts": "typescript", ".tsx": "typescript", ".java": "java",
    ".go": "go", ".rb": "ruby", ".rs": "rust", ".c": "c", ".h": "c",
    ".cpp": "cpp", ".hpp": "cpp", ".cs": "csharp", ".php": "php",
}

EXCLUDE_DIRS = {".git", "node_modules", "venv", ".venv", "__pycache__",
                "dist", "build", ".next", "target", "vendor", ".idea", ".mypy_cache"}


EXCLUDE_FILENAMES = {"package-lock.json", "yarn.lock", "poetry.lock"}
EXCLUDE_PATTERNS = re.compile(r"\.min\.(js|css)$|\.d\.ts$|_pb2\.py$")


def discover_files(repo_dir: Path, max_file_kb: int = 500):
    files = []
    for root, dirs, filenames in os.walk(repo_dir):
        dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS and not d.startswith(".")]
        for fn in filenames:
            if fn in EXCLUDE_FILENAMES or EXCLUDE_PATTERNS.search(fn):
                continue
            ext = Path(fn).suffix.lower()
            if ext not in CODE_ONLY_MAP:
                continue
            full = Path(root) / fn
            try:
                if full.stat().st_size > max_file_kb * 1024:
                    continue
            except OSError:
                continue
            files.append(full)
    return files

In [5]:

discovered = discover_files(repo_path)
print(f"Discovered {len(discovered)} chunkable files")
from collections import Counter
print(Counter(f.suffix for f in discovered).most_common())

Discovered 21 chunkable files
[('.py', 21)]


#### Chunking

In [6]:
@dataclass
class Chunk:
    repo: str
    file_path: str
    language: str
    symbol_type: str   # "function" | "class" | "method" | "block" | "file"
    symbol_name: str
    start_line: int
    end_line: int
    content: str
    char_count: int
    chunk_id: str = ""

    def __post_init__(self):
        if not self.chunk_id:
            raw = f"{self.repo}:{self.file_path}:{self.symbol_name}:{self.start_line}-{self.end_line}"
            self.chunk_id = hashlib.sha1(raw.encode()).hexdigest()[:16]


GENERIC_FUNC_PATTERNS = {
    "javascript": re.compile(r"^\s*(export\s+)?(async\s+)?function\s+(\w+)|^\s*(export\s+)?class\s+(\w+)|^\s*const\s+(\w+)\s*=\s*(async\s*)?\("),
    "typescript": re.compile(r"^\s*(export\s+)?(async\s+)?function\s+(\w+)|^\s*(export\s+)?class\s+(\w+)|^\s*const\s+(\w+)\s*=\s*(async\s*)?\("),
    "java": re.compile(r"^\s*(public|private|protected)?\s*(static\s+)?[\w<>\[\]]+\s+(\w+)\s*\("),
    "go": re.compile(r"^\s*func\s+(\(\w+\s+\*?\w+\)\s+)?(\w+)\s*\("),
    "ruby": re.compile(r"^\s*def\s+(\w+)|^\s*class\s+(\w+)"),
    "rust": re.compile(r"^\s*(pub\s+)?fn\s+(\w+)|^\s*(pub\s+)?struct\s+(\w+)"),
    "c": re.compile(r"^\s*[\w\*]+\s+(\w+)\s*\([^;]*\)\s*\{"),
    "cpp": re.compile(r"^\s*[\w\*:<>]+\s+(\w+)\s*\([^;]*\)\s*\{"),
    "csharp": re.compile(r"^\s*(public|private|protected)?\s*(static\s+)?[\w<>\[\]]+\s+(\w+)\s*\("),
    "php": re.compile(r"^\s*function\s+(\w+)|^\s*class\s+(\w+)"),
}

MAX_CHUNK_CHARS = 3000

def split_oversized(chunk: Chunk, max_chars: int = MAX_CHUNK_CHARS):
    if chunk.char_count <= max_chars:
        return [chunk]
    lines = chunk.content.splitlines()
    out, buf, buf_start = [], [], chunk.start_line
    cur_len = 0
    for i, line in enumerate(lines):
        buf.append(line)
        cur_len += len(line) + 1
        if cur_len >= max_chars:
            src = "\n".join(buf)
            out.append(Chunk(chunk.repo, chunk.file_path, chunk.language, chunk.symbol_type,
                              f"{chunk.symbol_name}_part{len(out)+1}", buf_start,
                              buf_start + len(buf) - 1, src, len(src)))
            buf, buf_start, cur_len = [], chunk.start_line + i + 1, 0
    if buf:
        src = "\n".join(buf)
        out.append(Chunk(chunk.repo, chunk.file_path, chunk.language, chunk.symbol_type,
                          f"{chunk.symbol_name}_part{len(out)+1}", buf_start,
                          buf_start + len(buf) - 1, src, len(src)))
    return out

def chunk_generic_lines(path: Path, repo_name: str, text: str, language: str, window: int = 60, overlap: int = 10):
    """Sliding-window line chunks with overlap. Fallback for languages/files
    without symbol-level parsing, or when a parse attempt fails."""
    lines = text.splitlines()
    chunks = []
    i, n = 0, len(lines)
    if n == 0:
        return chunks
    while i < n:
        end = min(i + window, n)
        src = "\n".join(lines[i:end])
        if src.strip():
            chunks.append(Chunk(repo_name, str(path), language, "block",
                                 f"lines_{i+1}-{end}", i + 1, end, src, len(src)))
        if end == n:
            break
        i += window - overlap
    return chunks


def chunk_python_file(path: Path, repo_name: str):
    text = path.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()
    try:
        tree = ast.parse(text)
    except SyntaxError:
        return chunk_generic_lines(path, repo_name, text, "python")

    chunks, covered = [], set()

    def node_source(node):
        start = node.lineno
        end = getattr(node, "end_lineno", start)
        covered.update(range(start, end + 1))
        return start, end, "\n".join(lines[start - 1:end])

    for node in ast.iter_child_nodes(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            start, end, src = node_source(node)
            chunks.append(Chunk(repo_name, str(path), "python", "function",
                                 node.name, start, end, src, len(src)))
        elif isinstance(node, ast.ClassDef):
            start, end, src = node_source(node)
            chunks.append(Chunk(repo_name, str(path), "python", "class",
                                 node.name, start, end, src, len(src)))
            for sub in node.body:
                if isinstance(sub, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    s2, e2, src2 = node_source(sub)
                    chunks.append(Chunk(repo_name, str(path), "python", "method",
                                         f"{node.name}.{sub.name}", s2, e2, src2, len(src2)))

    leftover = [i + 1 for i in range(len(lines)) if (i + 1) not in covered]
    if leftover:
        start, end = min(leftover), max(leftover)
        src = "\n".join(lines[start - 1:end])
        if src.strip():
            chunks.append(Chunk(repo_name, str(path), "python", "block",
                                 "module_level", start, end, src, len(src)))
    return chunks


def chunk_generic_symbols(path: Path, repo_name: str, language: str):
    text = path.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()
    pattern = GENERIC_FUNC_PATTERNS.get(language)
    if pattern is None:
        return chunk_generic_lines(path, repo_name, text, language)

    starts = [i for i, line in enumerate(lines) if pattern.search(line)]
    if not starts:
        return chunk_generic_lines(path, repo_name, text, language)

    chunks = []
    for idx, start in enumerate(starts):
        end = starts[idx + 1] - 1 if idx + 1 < len(starts) else len(lines) - 1
        while end > start and not lines[end].strip():
            end -= 1
        src = "\n".join(lines[start:end + 1])
        m = pattern.search(lines[start])
        name = next((g for g in m.groups() if g and re.match(r"^\w+$", g)), "anonymous")
        chunks.append(Chunk(repo_name, str(path), language, "function",
                             name, start + 1, end + 1, src, len(src)))
    return chunks


def chunk_file(path: Path, repo_name: str):
    language = CODE_ONLY_MAP.get(path.suffix.lower(), "text")
    if language == "python":
        return chunk_python_file(path, repo_name)
    if language in GENERIC_FUNC_PATTERNS:
        return chunk_generic_symbols(path, repo_name, language)
    text = path.read_text(encoding="utf-8", errors="ignore")
    return chunk_generic_lines(path, repo_name, text, language)


def chunk_repo(repo_dir: Path, repo_name: str):
    all_chunks = []
    for f in discover_files(repo_dir):
        for c in chunk_file(f,repo_name):
            all_chunks.extend(split_oversized(c))
    return all_chunks


In [7]:
import hashlib

chunks = chunk_repo(repo_path, repo_name)

print(f"Total chunks: {len(chunks)}")
from collections import Counter
print("By symbol_type:", Counter(c.symbol_type for c in chunks))
print("By language:   ", Counter(c.language for c in chunks))
print(f"Avg chunk size: {sum(c.char_count for c in chunks) / len(chunks):.0f} chars")

print("\nSample chunks:")
for c in chunks[:20]:
    print(f"  [{c.language:10}] {c.symbol_type:8} {c.symbol_name:25} "
          f"{Path(c.file_path).name}:{c.start_line}-{c.end_line}")


Total chunks: 122
By symbol_type: Counter({'method': 37, 'block': 34, 'function': 30, 'class': 21})
By language:    Counter({'python': 122})
Avg chunk size: 1095 chars

Sample chunks:
  [python    ] class    QueryRequest              api_server.py:43-47
  [python    ] class    TokenVerifyRequest        api_server.py:50-51
  [python    ] class    AppState                  api_server.py:58-65
  [python    ] function get_optional_session      api_server.py:77-82
  [python    ] function require_admin             api_server.py:85-92
  [python    ] function lifespan_part1            api_server.py:100-171
  [python    ] function lifespan_part2            api_server.py:172-197
  [python    ] function login_page                api_server.py:225-232
  [python    ] function verify_token              api_server.py:236-259
  [python    ] function get_me                    api_server.py:263-268
  [python    ] function logout                    api_server.py:272-275
  [python    ] function query_rag 

In [8]:
OUT_PATH = f"./temp/repos/{repo_name}/chunks-{repo_name}.jsonl"

with open(OUT_PATH, "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(asdict(c)) + "\n")

print(f"Wrote {len(chunks)} chunks to {OUT_PATH}")


Wrote 122 chunks to ./temp/repos/Hybrid-Search-RAG/chunks-Hybrid-Search-RAG.jsonl


#### Embedding and Db

In [9]:
import os
from dataclasses import asdict
from dotenv import load_dotenv
from astrapy import DataAPIClient
from astrapy.constants import VectorMetric
from sentence_transformers import SentenceTransformer
from astrapy.info import CollectionDefinition

load_dotenv()

# Load the embedding model locally (runs on your machine, free, no API calls)
model = SentenceTransformer("BAAI/bge-large-en-v1.5")

# Initialize the client
client = DataAPIClient()
db = client.get_database(
    api_endpoint=os.getenv("API_ENDPOINT"),
    token=os.getenv("API_TOKEN"),
)

definition = (
    CollectionDefinition.builder()
    .with_vector_dimension(1024)
    .with_vector_metric(VectorMetric.COSINE)
    .build()
)

# Drop the old collection if it exists — it was created with a `service` block,
# which is incompatible with bringing your own vectors. Must recreate clean.
if "repo_context" in db.list_collection_names():
    db.drop_collection("repo_context")

# Create collection WITHOUT a service block — no Astra-side embedding provider needed
collection = db.create_collection(
    "repo_context",
    definition=definition
)

# Compute embeddings locally
texts = [c.content for c in chunks]
vectors = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=32,
).tolist()

# Prepare documents with $vector (pre-computed), not $vectorize
documents = [{"_id": c.chunk_id, "$vector": vec, **asdict(c)} for vec, c in zip(vectors, chunks)]

# Insert chunks — no embedding provider call per-batch anymore, so no timeouts,
# can use a larger batch size and don't need retry logic for provider timeouts
batch_size = 50
all_inserted = []
for i in range(0, len(documents), batch_size):
    batch = documents[i:i + batch_size]
    result = collection.insert_many(batch, request_timeout_ms=30000)
    all_inserted.extend(result.inserted_ids)
    print(f"Batch {i // batch_size + 1}: inserted {len(result.inserted_ids)}")

print(f"\nSuccessfully inserted {len(all_inserted)} chunks into Astra DB!")
print(f"Collections in Astra DB: {db.list_collection_names()}")

/home/shreerag/Desktop/shreeragkh/Repo-context-copilot/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 4/4 [02:55<00:00, 43.76s/it]


Batch 1: inserted 50
Batch 2: inserted 50
Batch 3: inserted 22

Successfully inserted 122 chunks into Astra DB!
Collections in Astra DB: ['repo_context']


In [10]:
import json
import logging
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any

import bm25s

logger = logging.getLogger(__name__)


class BM25Retriever:
    """
    Wraps bm25s.BM25 to support:
      - building an index from a list of chunk dicts or dataclasses (text + metadata)
      - persisting the index and metadata to disk, scoped per repo via index_dir
      - reloading without re-tokenizing the corpus
      - querying with scores, metadata, and chunk_id attached (for RRF fusion
        against vector search results keyed on the same chunk_id)
    """

    def __init__(self, index_dir: str | Path = "bm25_index"):
        self.index_dir = Path(index_dir)
        self.retriever: bm25s.BM25 | None = None
        self.corpus: list[str] = []
        self.metadata: list[dict[str, Any]] = []

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------
    @staticmethod
    def _to_dicts(chunks: list[Any]) -> list[dict[str, Any]]:
        """Accept a list of dicts or dataclass instances (e.g. Chunk) transparently."""
        return [asdict(c) if is_dataclass(c) else c for c in chunks]

    # ------------------------------------------------------------------
    # Build
    # ------------------------------------------------------------------
    def build(self, chunks: list[Any], text_key: str = "content") -> None:
        """
        Build the BM25 index from scratch.

        Args:
            chunks: list of dicts or dataclass instances, each containing at
                    least `text_key` (e.g. the Chunk dataclass's `content` field,
                    and ideally a `chunk_id` field for fusion with vector results).
                    All other keys are stored as metadata and returned
                    alongside results at query time.
            text_key: the dict key holding the chunk's raw text. Defaults to
                    "content" to match the project's Chunk dataclass.
        """
        chunk_dicts = self._to_dicts(chunks)
        if not chunk_dicts:
            raise ValueError("Cannot build BM25 index from an empty chunk list.")

        self.corpus = [c[text_key] for c in chunk_dicts]
        self.metadata = [{k: v for k, v in c.items() if k != text_key} for c in chunk_dicts]

        logger.info("Tokenizing %d chunks for BM25 indexing...", len(self.corpus))
        tokens = bm25s.tokenize(self.corpus, show_progress=False)

        self.retriever = bm25s.BM25()
        self.retriever.index(tokens, show_progress=False)
        logger.info("BM25 index built with %d documents.", len(self.corpus))

    # ------------------------------------------------------------------
    # Incremental-ish rebuild (bm25s has no true incremental add;
    # this re-tokenizes the full corpus with new chunks appended)
    # ------------------------------------------------------------------
    def add(self, chunks: list[Any], text_key: str = "content") -> None:
        """
        Append new chunks and rebuild the index. bm25s does not support
        true incremental indexing, so this re-indexes the full corpus.
        Fine for periodic batch updates (e.g. re-ingesting a repo); avoid
        calling this per-request.
        """
        chunk_dicts = self._to_dicts(chunks)
        new_texts = [c[text_key] for c in chunk_dicts]
        new_meta = [{k: v for k, v in c.items() if k != text_key} for c in chunk_dicts]

        self.corpus.extend(new_texts)
        self.metadata.extend(new_meta)

        logger.info("Rebuilding BM25 index with %d total documents...", len(self.corpus))
        tokens = bm25s.tokenize(self.corpus, show_progress=False)
        self.retriever = bm25s.BM25()
        self.retriever.index(tokens, show_progress=False)

    # ------------------------------------------------------------------
    # Persistence
    # ------------------------------------------------------------------
    def save(self) -> None:
        """Persist the BM25 index, corpus, and metadata to self.index_dir."""
        if self.retriever is None:
            raise RuntimeError("No index to save. Call build() first.")

        self.index_dir.mkdir(parents=True, exist_ok=True)

        # bm25s handles the index + corpus itself
        self.retriever.save(str(self.index_dir), corpus=self.corpus)

        # metadata isn't tracked by bm25s, so store it ourselves
        meta_path = self.index_dir / "metadata.json"
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(self.metadata, f)

        logger.info("Saved BM25 index and metadata to %s", self.index_dir)

    def load(self) -> None:
        """Load a previously saved index, corpus, and metadata from disk."""
        if not self.index_dir.exists():
            raise FileNotFoundError(f"No index found at {self.index_dir}")

        self.retriever = bm25s.BM25.load(str(self.index_dir), load_corpus=True)

        meta_path = self.index_dir / "metadata.json"
        if meta_path.exists():
            with open(meta_path, "r", encoding="utf-8") as f:
                self.metadata = json.load(f)
        else:
            logger.warning("No metadata.json found at %s; metadata will be empty.", self.index_dir)
            self.metadata = [{} for _ in range(len(self.retriever.corpus))]

        self.corpus = [doc["text"] if isinstance(doc, dict) else doc for doc in self.retriever.corpus]

        # Detach the corpus from the bm25s object so retrieve() always
        # returns plain indices rather than document text/dicts. This keeps
        # query() lookups simple and correct even with duplicate chunk text.
        self.retriever.corpus = None

        logger.info("Loaded BM25 index with %d documents from %s", len(self.corpus), self.index_dir)

    # ------------------------------------------------------------------
    # Query
    # ------------------------------------------------------------------
    def query(self, query_text: str, k: int = 10) -> list[dict[str, Any]]:
        """
        Retrieve top-k chunks for a query.

        Returns:
            list of dicts: {"text": ..., "score": ..., "chunk_id": ..., "metadata": {...}}
            sorted by descending BM25 score. `chunk_id` is lifted out of metadata
            (if present) to the top level so it can be joined directly against
            vector search results in an RRF fusion step.
        """
        if self.retriever is None:
            raise RuntimeError("Index not built or loaded. Call build() or load() first.")

        k = min(k, len(self.corpus))
        if k == 0:
            return []

        query_tokens = bm25s.tokenize(query_text, show_progress=False)
        doc_indices, scores = self.retriever.retrieve(query_tokens, k=k, show_progress=False)

        results = []
        for idx, score in zip(doc_indices[0], scores[0]):
            idx = int(idx)
            meta = self.metadata[idx] if idx < len(self.metadata) else {}
            results.append({
                "text": self.corpus[idx],
                "score": float(score),
                "chunk_id": meta.get("chunk_id"),
                "metadata": meta,
            })
        return results

    def __len__(self) -> int:
        return len(self.corpus)

In [11]:
bm25=BM25Retriever(index_dir=f"./bm25_index/{repo_name}")
bm25.build(chunks)
bm25.save()

#### Retreiever

In [12]:
from typing import List, Dict, Any


class VectorRetriever:
    """Handles query-based semantic retrieval from the Astra DB vector store."""

    def __init__(self, collection, model):
        """
        Initialize the retriever pipeline for Astra DB.

        Args:
            collection: an astrapy Collection with pre-computed $vector fields
                        (see ingestion pipeline — chunks were embedded locally
                        with sentence-transformers and inserted as $vector).
            model: the same SentenceTransformer instance used at ingestion time.
                   Must match exactly, or query/document vectors won't be comparable.
        """
        self.collection = collection
        self.model = model

    def query(self, query_text: str, k: int = 5) -> List[Dict[str, Any]]:
        """Helper to match the query interface of other retrievers (e.g. BM25Retriever)."""
        return self.retrieve(query_text, top_k=k)

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant chunks for a query via vector similarity search.

        Args:
            query: query from the user
            top_k: number of top results to return
            score_threshold: minimum similarity score threshold (0-1, cosine)

        Returns:
            List of dicts: {"chunk_id", "text", "content", "metadata",
            "score", "rank"} — shaped to match BM25Retriever.query() output
            so both can be merged directly in RRF fusion.
        """
        try:
            query_vector = self.model.encode(
                [query], normalize_embeddings=True
            ).tolist()[0]

            results = self.collection.find(
                sort={"$vector": query_vector},
                limit=top_k,
                include_similarity=True,
            )

            retrieved_docs = []
            for i, doc in enumerate(results):
                similarity_score = doc.get("$similarity", 0.0)
                if similarity_score < score_threshold:
                    continue

                content = doc.get("content", "")
                metadata = {
                    k: v for k, v in doc.items()
                    if k not in ("_id", "$vector", "$similarity", "content")
                }

                retrieved_docs.append({
                    "id": doc.get("_id"),
                    "chunk_id": doc.get("chunk_id", doc.get("_id")),
                    "text": content,
                    "content": content,
                    "metadata": metadata,
                    "score": similarity_score,
                    "similarity_score": similarity_score,
                    "rank": i + 1,
                })

            print(f"Retrieved documents: {len(retrieved_docs)} documents (after filtering)")
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


# Usage
vector_retrieval = VectorRetriever(collection, model)  # `model` = your SentenceTransformer instance

#### HybridSearch

In [13]:
from __future__ import annotations
import logging
import time
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeoutError
from typing import Any, Callable

logger = logging.getLogger(__name__)


class HybridSearchError(Exception):
    """Raised only when BOTH retrievers fail — total retrieval failure."""


def _reciprocal_rank_fusion(
    bm25_results: list[dict],
    vector_results: list[dict],
    bm25_weight: float,
    vector_weight: float,
    rrf_k: int = 60,
    id_key: str = "chunk_id",
) -> list[dict]:

    fused_docs = {}

    def get_id(doc):
        # chunk_id is exposed at the TOP LEVEL by both BM25Retriever and
        # VectorRetriever (see their query() implementations), so check
        # there first. Fall back to metadata, then raw text as a last resort
        # for any retriever that doesn't provide a stable chunk_id.
        if doc.get(id_key):
            return str(doc[id_key])
        meta = doc.get("metadata") or {}
        if id_key in meta and meta[id_key]:
            return str(meta[id_key])
        text = doc.get("text") or doc.get("content") or ""
        return text.strip()

    for rank, doc in enumerate(bm25_results):
        doc_id = get_id(doc)
        text = doc.get("text") or doc.get("content") or ""
        metadata = doc.get("metadata") or {}
        score = doc.get("score", 0.0)

        fused_docs[doc_id] = {
            "chunk_id": doc.get(id_key) or doc_id,
            "text": text,
            "metadata": metadata,
            "bm25_score": score,
            "vector_score": 0.0,
            "bm25_rrf": bm25_weight * (1.0 / (rrf_k + (rank + 1))),
            "vector_rrf": 0.0,
        }

    for rank, doc in enumerate(vector_results):
        doc_id = get_id(doc)
        text = doc.get("text") or doc.get("content") or ""
        metadata = doc.get("metadata") or {}
        score = doc.get("similarity_score") or doc.get("score") or 0.0

        if doc_id in fused_docs:
            fused_docs[doc_id]["vector_score"] = score
            fused_docs[doc_id]["vector_rrf"] = vector_weight * (1.0 / (rrf_k + (rank + 1)))
            if not fused_docs[doc_id]["metadata"] and metadata:
                fused_docs[doc_id]["metadata"] = metadata
            if not fused_docs[doc_id]["text"] and text:
                fused_docs[doc_id]["text"] = text
        else:
            fused_docs[doc_id] = {
                "chunk_id": doc.get(id_key) or doc_id,
                "text": text,
                "metadata": metadata,
                "bm25_score": 0.0,
                "vector_score": score,
                "bm25_rrf": 0.0,
                "vector_rrf": vector_weight * (1.0 / (rrf_k + (rank + 1))),
            }

    output = []
    for doc_id, info in fused_docs.items():
        fused_score = info["bm25_rrf"] + info["vector_rrf"]
        output.append({
            "chunk_id": info["chunk_id"],
            "text": info["text"],
            "metadata": info["metadata"],
            "fused_score": fused_score,
            "bm25_score": info["bm25_score"],
            "vector_score": info["vector_score"],
        })

    output.sort(key=lambda x: x["fused_score"], reverse=True)
    return output


class HybridSearch:
    """Handles query-based hybrid search: BM25 + vector retrieval fused via RRF."""

    def __init__(self, bm25_retriever, vector_retriever):
        self.bm25_retriever = bm25_retriever
        self.vector_retriever = vector_retriever

    def hybrid_retrieval(
        self,
        query_text: str,
        k: int = 10,
        fetch_k: int = 25,
        bm25_weight: float = 0.4,
        vector_weight: float = 0.6,
        rrf_k: int = 60,
        id_key: str = "chunk_id",
        metadata_filter: Callable[[dict], bool] | None = None,
        timeout_s: float = 15.0,
    ) -> list[dict[str, Any]]:
        """
        Runs BM25 and vector retrieval in parallel, fuses with RRF, and returns top-k.

        Args:
            query_text: user's query, e.g. "how does the auth middleware work?"
            k: number of results to return after fusion.
            fetch_k: candidates pulled from EACH retriever before fusion.
            bm25_weight / vector_weight: RRF weighting between the two signals.
            rrf_k: RRF damping constant (60 is the standard default).
            id_key: field used as the stable dedup key across both retrievers.
                    Defaults to "chunk_id", set on every Chunk at ingestion time
                    and returned at the top level by both retrievers.
            metadata_filter: optional predicate applied after fusion, e.g.
                    lambda m: m.get("language") == "python".
            timeout_s: max seconds to wait for EACH retriever before treating
                    it as failed and falling back to the other.

        Returns:
            List of {"chunk_id", "text", "metadata", "fused_score",
            "bm25_score", "vector_score"} sorted by fused_score descending.

        Raises:
            HybridSearchError if both retrievers fail.
        """
        start = time.monotonic()
        bm25_results, vector_results = self._run_retrievers_with_fallback(
            query_text, fetch_k, timeout_s
        )

        fused = _reciprocal_rank_fusion(
            bm25_results, vector_results, bm25_weight, vector_weight, rrf_k, id_key
        )

        if metadata_filter is not None:
            fused = [r for r in fused if metadata_filter(r.get("metadata", {}))]
        results = fused[:k]

        logger.info(
            "hybrid_search query=%r bm25_hits=%d vector_hits=%d fused=%d returned=%d latency_ms=%.0f",
            query_text, len(bm25_results), len(vector_results), len(fused), len(results),
            (time.monotonic() - start) * 1000,
        )
        return results

    def _run_retrievers_with_fallback(
        self, query_text: str, fetch_k: int, timeout_s: float
    ) -> tuple[list[dict], list[dict]]:
        """Run both retrievers concurrently; a failure/timeout in one degrades
        gracefully to results from the other instead of raising."""

        def safe_call(fn, name: str) -> list[dict]:
            try:
                return fn(query_text, k=fetch_k)
            except Exception:
                logger.exception("Retriever %s failed", name)
                return []

        with ThreadPoolExecutor(max_workers=2) as executor:
            bm25_future = executor.submit(safe_call, self.bm25_retriever.query, "bm25")
            vector_future = executor.submit(safe_call, self.vector_retriever.query, "vector")

            try:
                bm25_results = bm25_future.result(timeout=timeout_s)
            except FutureTimeoutError:
                logger.warning("BM25 retriever timed out after %.1fs", timeout_s)
                bm25_results = []

            try:
                vector_results = vector_future.result(timeout=timeout_s)
            except FutureTimeoutError:
                logger.warning("Vector retriever timed out after %.1fs", timeout_s)
                vector_results = []

        if not bm25_results and not vector_results:
            raise HybridSearchError(f"Both retrievers failed or timed out for query: {query_text!r}")
        return bm25_results, vector_results


# Usage — bm25_retriever from BM25Retriever, vector_retriever from VectorRetriever
hybrid_search = HybridSearch(bm25, vector_retrieval)
hybrid_search.hybrid_retrieval("how does the authentication middleware work?")

Retrieved documents: 25 documents (after filtering)


[{'chunk_id': 'ca2e416d26e7da24',
  'text': '"""\nFirebase Authentication module.\n\nVerifies Firebase ID tokens using Google\'s public keys (no service account\nneeded) and manages lightweight server-side sessions.\n"""\n\nimport os\nimport uuid\nimport logging\nfrom datetime import datetime, timedelta, timezone\nfrom typing import Any\n\nfrom google.oauth2 import id_token as google_id_token\nfrom google.auth.transport import requests as google_requests\n\nlogger = logging.getLogger(__name__)\n\nADMIN_EMAIL = os.getenv("ADMIN_EMAIL", "shreerag99@gmail.com")\nFIREBASE_PROJECT_ID = os.getenv("FIREBASE_PROJECT_ID", "hybrid-search-rag")\n\n# ---------------------------------------------------------------------------\n# In-memory session store  (single-server; sufficient for this app)\n# ---------------------------------------------------------------------------\n_sessions: dict[str, dict[str, Any]] = {}\n\n\ndef verify_firebase_token(token: str) -> dict[str, Any]:\n    """\n    Verify a F

#### Reranker

In [14]:
from __future__ import annotations
 
import logging
import time
from typing import Any, Protocol
 
logger = logging.getLogger(__name__)
 
DEFAULT_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
# Stronger, slower alternative: "BAAI/bge-reranker-base" or "BAAI/bge-reranker-large"
 
 
class ScoringBackend(Protocol):
    """Minimal interface a reranking backend must satisfy."""
    def predict(self, pairs: list[tuple[str, str]]) -> list[float]: ...
 
 
class RerankerError(Exception):
    """Raised when reranking fails and no safe fallback is possible."""
 
 
class Reranker:
    def __init__(
        self,
        model_name: str = DEFAULT_MODEL,
        batch_size: int = 32,
        device: str | None = None,
        backend: ScoringBackend | None = None,
    ):
        """
        Args:
            model_name: HuggingFace cross-encoder model id. Ignored if
                        `backend` is supplied.
            batch_size: pairs per forward pass. Tune to your GPU/CPU memory.
            device: "cuda", "cpu", or None to let sentence-transformers pick.
            backend: inject a custom scoring backend (e.g. a Cohere Rerank
                     wrapper) instead of loading a local model. Must expose
                     .predict(list[(query, doc_text)]) -> list[float].
        """
        self.batch_size = batch_size
        self.model_name = model_name
 
        if backend is not None:
            self.backend = backend
        else:
            self.backend = self._load_local_model(model_name, device)
 
    @staticmethod
    def _load_local_model(model_name: str, device: str | None):
        try:
            from sentence_transformers import CrossEncoder
        except ImportError as e:
            raise ImportError(
                "sentence-transformers is required for local reranking. "
                "Install with: pip install sentence-transformers --break-system-packages"
            ) from e
 
        logger.info("Loading cross-encoder reranker model: %s", model_name)
        model = CrossEncoder(model_name, device=device)
        return model
 
    def rerank(
        self,
        query: str,
        candidates: list[dict[str, Any]],
        top_n: int = 5,
        min_score: float | None = None,
        text_key: str = "text",
        fallback_on_error: bool = True,
    ) -> list[dict[str, Any]]:
        """
        Score each candidate against the query and return the top_n,
        re-sorted by cross-encoder relevance score.
 
        Args:
            query: the user query.
            candidates: list of dicts (as returned by hybrid_search), each
                        containing at least `text_key`.
            top_n: number of results to return after reranking.
            min_score: optional threshold; candidates scoring below this
                       are dropped even if within top_n. Use this to avoid
                       feeding clearly-irrelevant context to the LLM when
                       retrieval had a bad day.
            text_key: dict key holding each candidate's text.
            fallback_on_error: if True and scoring fails, return the
                       original candidates truncated to top_n rather than
                       raising — keeps the pipeline degrading gracefully
                       instead of hard-failing generation.
 
        Returns:
            List of candidate dicts (original fields preserved) with an
            added "rerank_score" key, sorted descending, length <= top_n.
        """
        if not candidates:
            return []
 
        start = time.monotonic()
        pairs = [(query, c[text_key]) for c in candidates]
 
        try:
            scores = self._score_in_batches(pairs)
        except Exception:
            logger.exception("Reranking failed for query=%r (%d candidates)", query, len(candidates))
            if fallback_on_error:
                logger.warning("Falling back to pre-rerank order (no cross-encoder scores applied).")
                return [{**c, "rerank_score": c.get("fused_score", 0.0)} for c in candidates[:top_n]]
            raise RerankerError(f"Reranking failed for query: {query!r}")
 
        scored = [
            {**cand, "rerank_score": float(score)}
            for cand, score in zip(candidates, scores)
        ]
        scored.sort(key=lambda c: c["rerank_score"], reverse=True)
 
        if min_score is not None:
            scored = [c for c in scored if c["rerank_score"] >= min_score]
 
        results = scored[:top_n]
 
        logger.info(
            "rerank query=%r candidates=%d returned=%d top_score=%.4f latency_ms=%.0f",
            query, len(candidates), len(results),
            results[0]["rerank_score"] if results else float("nan"),
            (time.monotonic() - start) * 1000,
        )
        return results
 
    def _score_in_batches(self, pairs: list[tuple[str, str]]) -> list[float]:
        scores: list[float] = []
        for i in range(0, len(pairs), self.batch_size):
            batch = pairs[i : i + self.batch_size]
            batch_scores = self.backend.predict(batch)
            scores.extend(float(s) for s in batch_scores)
        return scores


In [15]:
re_ranker=Reranker()
re_ranker

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1897.58it/s]


#### RAG PipeLine

In [16]:
import re
import markdown
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage

load_dotenv()

# Initialize LLM with fallback if GROQ_API_KEY is not configured
if os.environ.get("GROQ_API_KEY"):
    llm = ChatGroq(model="qwen/qwen3.6-27b", reasoning_format="hidden", temperature=0.1, max_tokens=4096)
else:
    print("GROQ_API_KEY not found in environment. Falling back to FakeMessagesListChatModel for mock responses.")

def ragPipeline(query, hybrid_search=None, reranker=None, llm=None, top_k=5, top_n=3, min_score=0.2, return_context=False):
    """
        RAG Pipeline with extra Features
        - Returns answer, sources, confidence_score and optionally full context.
    """
    # Auto-align in case llm is passed positionally as the third parameter (reranker)
    if reranker is not None and (hasattr(reranker, 'invoke') and not hasattr(reranker, 'rerank')):
        llm = reranker
        reranker = None

    if hybrid_search is None:
        hybrid_search = globals().get('hybrid_search')
        if hybrid_search is None:
            raise ValueError("No hybrid_search instance provided or found in global scope.")
            
    if llm is None:
        llm = globals().get('llm')
        if llm is None:
            raise ValueError("No LLM instance provided or found in global scope.")

    if reranker is None:
        reranker = globals().get('re_ranker') or globals().get('reranker')

    # Call hybrid_retrieval
    results = hybrid_search.hybrid_retrieval(query, k=top_k)
    
    # RRF weights default to 0.4 (bm25) and 0.6 (vector), rrf_k defaults to 60.
    # Calculate max possible theoretical RRF score for normalization to [0, 1].
    bm25_weight = 0.4
    vector_weight = 0.6
    rrf_k = 60
    max_rrf_score = (bm25_weight + vector_weight) / (rrf_k + 1)
    
    # Filter results by min_score using normalized fused_score
    results = [doc for doc in results if (doc.get('fused_score', 0.0) / max_rrf_score) >= min_score]
    
    if not results:
        return {'answer': 'No relevant answer found.', 'sources': [], 'confidence': 0.0, 'context': ""}
    
    # 2. rerank down to a tight, high-precision set if reranker is provided
    if reranker is not None:
        results = reranker.rerank(query, results, top_n=top_n)
    
    if not results:
        return {'answer': 'No relevant answer found after rerank.', 'sources': [], 'confidence': 0.0, 'context': ""}

    # Prepare context and sources using normalized scores
    context = "\n\n".join([doc['text'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc.get('fused_score', 0.0) / max_rrf_score,
        'preview': doc['text'][:120] + '...'
    } for doc in results]

    confidence = max(doc.get('fused_score', 0.0) / max_rrf_score for doc in results)

    ## Generate the answer
    prompt = f"""You are a helpful assistant. Use the following context to answer the question.
                Provide ONLY the direct answer. Do not include any conversational filler, introductory text, explanation, or polite remarks.

                Context:
                {context}

                Question: {query}

                Answer:"""
    response = llm.invoke([prompt])
    content = response.content

    # Remove thinking/thought blocks (e.g. <think>...</think>)
    content = re.sub(r'<think>.*?</think>', '', content, flags=re.DOTALL)

    # Strip markdown formatting by converting to HTML and removing all tags
    html = markdown.markdown(content)
    plain_text = re.sub(r'<[^>]*>', '', html)

    output = {
        'answer': plain_text.strip(),
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context
    return output


In [17]:
result = ragPipeline("where i have used auth", hybrid_search=hybrid_search, reranker=re_ranker, llm=llm)
print(result['answer'])
print(result['confidence'])


Retrieved documents: 25 documents (after filtering)
GET /auth/login — Firebase Google Sign-In page
POST /auth/verify — verify Firebase token, create session
GET /auth/me — get current user info from session
POST /auth/logout — destroy session
POST /api/query — uses get_optional_session dependency to check admin status
Auth dependencies/functions: get_optional_session, require_admin, verify_firebase_token, create_session, get_session, delete_session, is_admin
0.9753846153846154
